In [1]:
from pathlib import Path
import json
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent
OUT = PROJECT_ROOT / "data" / "processed" / "tutor"
OUT.mkdir(parents=True, exist_ok=True)

DOCS = PROJECT_ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)

print("Tutor out:", OUT)

Tutor out: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\tutor


In [2]:
RULES = {
    "R1_never_solution_first": True,
    "R2_diagnose_before_intervene": True,
    "R3_one_intervention_target": True,
    "R4_prefer_learner_reasoning": True,
    "R5_escalate_hints_gradually": True,
    "R6_retest_after_intervention": True,
    "R7_evidence_for_mastery": True,
    "R8_transfer_before_mastery": True,
    "R9_repeated_failure_remediate_not_harder": True,
    "R10_n08_trace_min_share": 0.70,
}

(OUT / "tutor_rules_v1.json").write_text(json.dumps(RULES, indent=2), encoding="utf-8")
print(RULES)

{'R1_never_solution_first': True, 'R2_diagnose_before_intervene': True, 'R3_one_intervention_target': True, 'R4_prefer_learner_reasoning': True, 'R5_escalate_hints_gradually': True, 'R6_retest_after_intervention': True, 'R7_evidence_for_mastery': True, 'R8_transfer_before_mastery': True, 'R9_repeated_failure_remediate_not_harder': True, 'R10_n08_trace_min_share': 0.7}


In [3]:
HINT_POLICY = {
    "levels": {
        "H0": "Socratic prompt only",
        "H1": "Conceptual pointer",
        "H2": "Strategic operation hint",
        "H3": "One micro-step only",
        "H4": "Stepwise solution with teach-back (never dump)"
    },
    "escalation": {
        "type": "hybrid_attempt_primary",
        "attempt_map": {1: "H0", 2: "H1", 3: "H2", 4: "H3", 5: "H4"},
        "silence_seconds_offer_hint": 90,
        "explicit_answer_request": "one_more_H2_then_stepwise_H4"
    }
}

(OUT / "hint_policy_v1.json").write_text(json.dumps(HINT_POLICY, indent=2), encoding="utf-8")

490

In [4]:
LEARNER_STATE_SCHEMA = {
    "learner_id": "string",
    "topic": "string",
    "misconception_id": "string|null",
    "intervention_id": "string|null",
    "hint_level": "H0|H1|H2|H3|H4",
    "attempt_count": "int",
    "session_mastery_status": "in_progress|passed|failed",
    "skill_mastery_status": "not_started|in_progress|mastered|needs_human_review",
    "error_history": "list",
    "confidence_last": "1|2|3|null",
    "engagement_state": "active|silent|frustrated|answer_demand|disconnected",
    "n08_trace_share_session": "float",
    "updated_at": "iso8601"
}

(OUT / "learner_state_schema_v1.json").write_text(
    json.dumps(LEARNER_STATE_SCHEMA, indent=2), encoding="utf-8"
)

524

In [6]:
from pathlib import Path
import json
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent
OUT = PROJECT_ROOT / "data" / "processed" / "tutor"
DOCS = PROJECT_ROOT / "docs"
OUT.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)

RULES = {
    "R1_never_solution_first": True,
    "R2_diagnose_before_intervene": True,
    "R3_one_intervention_target": True,
    "R4_prefer_learner_reasoning": True,
    "R5_escalate_hints_gradually": True,
    "R6_retest_after_intervention": True,
    "R7_evidence_for_mastery": True,
    "R8_transfer_before_mastery": True,
    "R9_repeated_failure_remediate_not_harder": True,
    "R10_n08_trace_min_share": 0.70,
}

HINT_POLICY = {
    "levels": {
        "H0": "Socratic prompt only",
        "H1": "Conceptual pointer",
        "H2": "Strategic operation hint",
        "H3": "One micro-step only",
        "H4": "Stepwise solution with teach-back (never dump)",
    },
    "escalation": {
        "type": "hybrid_attempt_primary",
        "attempt_map": {1: "H0", 2: "H1", 3: "H2", 4: "H3", 5: "H4"},
        "silence_seconds_offer_hint": 90,
        "explicit_answer_request": "one_more_H2_then_stepwise_H4",
    },
}

MASTERY_HIERARCHY = {
    "session_mastery": {
        "source": "N08",
        "correct_required": 4,
        "items_presented": 5,
        "unseen_only": True,
        "structure_type_match": True,
        "sittings_required_for_skill": 2,
    },
    "skill_mastery": {
        "source": "N09",
        "min_sessions_passed": 2,
        "max_hints_on_final_session": 1,
        "no_target_misconception_recurrence": True,
        "require_transfer_item": True,
    },
}

LEARNER_STATE_SCHEMA = {
    "learner_id": "string",
    "topic": "string",
    "misconception_id": "string|null",
    "intervention_id": "string|null",
    "hint_level": "H0|H1|H2|H3|H4",
    "attempt_count": "int",
    "session_mastery_status": "in_progress|passed|failed",
    "skill_mastery_status": "not_started|in_progress|mastered|needs_human_review",
    "error_history": "list",
    "confidence_last": "1|2|3|null",
    "engagement_state": "active|silent|frustrated|answer_demand|disconnected",
    "n08_trace_share_session": "float",
    "updated_at": "iso8601",
}

STATE_MACHINE = {
    "states": [
        "await_attempt", "diagnose", "select_intervention", "deliver_hint",
        "reassess", "session_mastery_check", "skill_mastery_check",
        "remediate", "human_review_flag", "disconnected",
    ],
    "transitions": [
        {"from": "await_attempt", "on": "learner_response", "to": "diagnose"},
        {"from": "diagnose", "on": "matched_misconception", "to": "select_intervention"},
        {"from": "diagnose", "on": "no_match", "to": "deliver_hint", "hint": "generic_fallback"},
        {"from": "select_intervention", "on": "n08_pattern", "to": "deliver_hint"},
        {"from": "deliver_hint", "on": "learner_retry", "to": "diagnose"},
        {"from": "reassess", "on": "session_pass", "to": "skill_mastery_check"},
        {"from": "reassess", "on": "session_fail", "to": "remediate"},
        {"from": "skill_mastery_check", "on": "mastered", "to": "await_attempt"},
        {"from": "skill_mastery_check", "on": "persistent_fail", "to": "human_review_flag"},
        {"from": "*", "on": "disconnect", "to": "disconnected"},
        {"from": "disconnected", "on": "reconnect", "to": "await_attempt", "resume": True},
    ],
}

# Save JSON artefacts
(OUT / "tutor_rules_v1.json").write_text(json.dumps(RULES, indent=2), encoding="utf-8")
(OUT / "hint_policy_v1.json").write_text(json.dumps(HINT_POLICY, indent=2), encoding="utf-8")
(OUT / "mastery_hierarchy_v1.json").write_text(json.dumps(MASTERY_HIERARCHY, indent=2), encoding="utf-8")
(OUT / "learner_state_schema_v1.json").write_text(json.dumps(LEARNER_STATE_SCHEMA, indent=2), encoding="utf-8")
(OUT / "tutor_state_machine_v1.json").write_text(json.dumps(STATE_MACHINE, indent=2), encoding="utf-8")

spec_md = f"""# Tutor Behaviour Specification v1

Generated: {datetime.now(timezone.utc).isoformat()}

## Audience
Young NSC Mathematics learner (Grade 10–12 range). No fixed age assumption.

## Core claim
Evidence-driven tutoring: DBE diagnostic errors → N06 → N07 → N08 → tutor action.

## Hard rules
{json.dumps(RULES, indent=2)}

## Hint policy
{json.dumps(HINT_POLICY, indent=2)}

## Mastery hierarchy
{json.dumps(MASTERY_HIERARCHY, indent=2)}

## LLM boundary
- LLM may classify free-text errors and render hint text at a rule-assigned level.
- LLM may not choose intervention, hint level, or mastery outcome.

## SA constraints
- Persist state across disconnect (load-shedding / data drop).
- Accept code-switching; respond in dominant language of learner input.
- Never shame; acknowledge frustration; reduce difficulty before continuing.

## Provenance
Every tutor event must log: session_event_id, intervention_id (or generic_fallback),
topic, misconception_id, hint_level, rule_ids_fired.
"""

(DOCS / "09_Tutor_Behaviour_Spec.md").write_text(spec_md, encoding="utf-8")
(OUT / "tutor_behaviour_spec_v1.md").write_text(spec_md, encoding="utf-8")

print("NOTEBOOK 09 artefacts written:")
for p in sorted(OUT.glob("*")):
    print(" -", p.name)
print(" -", DOCS / "09_Tutor_Behaviour_Spec.md")

NOTEBOOK 09 artefacts written:
 - hint_policy_v1.json
 - learner_state_schema_v1.json
 - mastery_hierarchy_v1.json
 - tutor_behaviour_spec_v1.md
 - tutor_rules_v1.json
 - tutor_state_machine_v1.json
 - c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\docs\09_Tutor_Behaviour_Spec.md


In [7]:
STATE_MACHINE = {
    "states": [
        "await_attempt",
        "diagnose",
        "select_intervention",
        "deliver_hint",
        "reassess",
        "session_mastery_check",
        "skill_mastery_check",
        "remediate",
        "human_review_flag",
        "disconnected"
    ],
    "transitions": [
        {"from": "await_attempt", "on": "learner_response", "to": "diagnose"},
        {"from": "diagnose", "on": "matched_misconception", "to": "select_intervention"},
        {"from": "diagnose", "on": "no_match", "to": "deliver_hint", "hint": "generic_fallback"},
        {"from": "select_intervention", "on": "n08_pattern", "to": "deliver_hint"},
        {"from": "deliver_hint", "on": "learner_retry", "to": "diagnose"},
        {"from": "reassess", "on": "session_pass", "to": "skill_mastery_check"},
        {"from": "reassess", "on": "session_fail", "to": "remediate"},
        {"from": "skill_mastery_check", "on": "mastered", "to": "await_attempt"},
        {"from": "skill_mastery_check", "on": "persistent_fail", "to": "human_review_flag"},
        {"from": "*", "on": "disconnect", "to": "disconnected"},
        {"from": "disconnected", "on": "reconnect", "to": "await_attempt", "resume": True}
    ]
}

(OUT / "tutor_state_machine_v1.json").write_text(json.dumps(STATE_MACHINE, indent=2), encoding="utf-8")
print("State machine saved")

State machine saved
